In [ ]:
import pandas as pd
from typing import List, Dict, Any


def computestats(records: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(records)

    agg = (
        df.groupby(["language", "deprel"])
        .agg(
            n_pairs           = ("dep_length", "count"),
            mean_dep_length   = ("dep_length", "mean"),
            median_dep_length = ("dep_length", "median"),
            mean_arity        = ("intervener_arity", "mean"),
            mean_subtree_size = ("intervener_subtree_size", "mean"),
            pct_heads_struct  = ("intervener_heads_structure", "mean"),
        )
        .reset_index()
    )

    for col in ["mean_dep_length", "median_dep_length", "mean_arity",
                "mean_subtree_size", "pct_heads_struct"]:
        agg[col] = agg[col].round(3)


    def top_pos(sub_df, n=3):
        counts = sub_df["intervener_upos"].value_counts()
        return ", ".join(f"{pos}({cnt})" for pos, cnt in counts.head(n).items())

    pos_info = (
        df.groupby(["language", "deprel"])
        .apply(top_pos)
        .reset_index()
        .rename(columns={0: "top_intervener_pos"})
    )

    stats = agg.merge(pos_info, on=["language", "deprel"])
    return stats


def printsummary(stats_df: pd.DataFrame):

    print("\n=== SUMMARY: Mean Intervener Arity by Dependency Type (all languages) ===")
    summary = (
        stats_df.groupby("deprel")["mean_arity"]
        .mean()
        .sort_values()
    )
    for dep, val in summary.items():
        bar = "█" * int(val * 10)
        print(f"  {dep:<10} {val:.3f}  {bar}")

    print("\n=== SUMMARY: Mean Subtree Size by Dependency Type ===")
    summary2 = (
        stats_df.groupby("deprel")["mean_subtree_size"]
        .mean()
        .sort_values()
    )
    for dep, val in summary2.items():
        bar = "█" * int(val * 4)
        print(f"  {dep:<10} {val:.3f}  {bar}")
